laboratory testのパネル分類

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import japanize_matplotlib
%matplotlib inline
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from IPython.display import display
import time

import sys
from decimal import Decimal, ROUND_HALF_UP
import re
#For exporting .pdf file with editable text
import matplotlib
matplotlib.rcParams['pdf.fonttype']=42
matplotlib.rcParams['ps.fonttype']=42

import collections

#!pip install MissForest
from missforest import MissForest
from sklearn.ensemble import RandomForestRegressor
import gc
from sklearn.impute import SimpleImputer

from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.decomposition import PCA

#!conda install -c conda-forge -y watermark
%load_ext watermark
%watermark -a 'Akari FUJIMOTO' -udnt -vmw -iv

In [ ]:
japanize_matplotlib.japanize()

print(plt.rcParams['font.family'])

In [ ]:
import math

In [ ]:
from scipy.stats import pearsonr

# 1. import laboratorytest

In [ ]:
#Import dataframe
fileDir = './ExportData/Preprocess/'
ipynbName = '260916_TMM-BAge_CommCohort67K-ver1_'
fileName = 'laboDF-rename-date.tsv'
tempDF = pd.read_csv(fileDir+ipynbName+fileName, sep='\t')

display(tempDF)
display(tempDF.describe(include='all'))
specificDF=tempDF #laboratory

In [ ]:
#推定される対応
spe_dict = {
    'hematology': [
        '桿状核球 [%]',
        '分葉核球 [%]',
        '赤芽球 [/100WBC]',
        'A-Lymphocyte [%]',
        'Pro-Myelo [%]',
        'Myeloblast [%]',
        'Myelo [%]',
        'Meta-Myelo [%]'],

    'clinical_test': [
        'NT-proBNP',
        'グリコアルブミン [%]',
        'ペプシノーゲンⅠ [ng/mL]',
        'ペプシノーゲンⅡ [ng/mL]',
        'PG1/PG2 比 ',
        '高感度心筋トロポニンT [ng/mL]'],

    'allergy': [
        # アレルゲン関連項目
        'オオアワガエリ [LC]',
        'ハルガヤ [LC]',
        'カモガヤ [LC]',
        'カモガヤ [UA/mL]', 'ブタクサ混合物Ⅰ [LC]', 'ヨモギ [LC]', 'ヨモギ [UA/mL] ',
        'スギ [LC]', 'スギ [UA/mL] ', 'スギ [UA/mL] 岩手', 'ヒノキ [LC]',
        'ハンノキ [LC]', 'シラカンバ [LC]', 'コナヒョウヒダニ [LC]', 
        'コナヒョウヒダニ [UA/mL] ', 'コナヒョウヒダニ [UA/mL] 岩手', 'ハウスダストⅠ [LC]', 
        'ペニシリウム [LC]', 'クラドスポリウム [LC]', 'カンジダ [LC]', 'カンジダ [UA/mL]', 
        'アルテルナリア [LC]', 'アスペルギルス [LC]', 'ネコ皮屑 [LC]', 'ネコ皮屑 [UA/mL]', 
        'イヌ皮屑 [LC]', '小麦 [LC]', '大豆 [LC]', '米 [LC]', 'マグロ [LC]', 'サケ [LC]', 
        'エビ [LC]', 'エビ [UA/mL]', 'カニ [LC]', 'チェダーチーズ [LC]', 'ミルク [LC]', 
        '牛肉 [LC]', '鶏肉 [LC]', '卵白 [LC]', '卵白 [UA/mL]', 'ソバ [LC]', 
        'ピーナッツ [LC]', 'ラテックス [LC]',
        '抗体濃度 [U/mL]',
        'IgE（非特異的IgE） [IU/mL]'],

    'kidney': [
        'アルブミン／Cre補正 [mg/g.cre]',
        'アルブミン定量値 [μg/mL]',
        'クレアチニン（尿中） [g/L]',
        'Na（ナトリウム）（尿中） [g/L]',
        'Cl（クロール）（尿中） [g/L]',
        'K（カリウム）（尿中） [g/L]',
        'シスタチンC［金コロイド凝集法］ [mg/L]',
        'GFR区分（シスタチンC）[ml/min/1.73㎡]'],
    
    'administrative': [
        '受診日',
        'イベントコード',
        'コースコード','検査会社']}

In [ ]:
#Save cohort metadata
fileDir = './ExportData/Preprocess/metabolome/'
ipynbName = '260924_TMM-BAge_PreprocessMetDF-CommCohort67K-ver1_'
fileName = 'demographicsDF_2k.tsv'
tempDF = pd.read_csv(fileDir+ipynbName+fileName, sep='\t')
tempDF = tempDF.set_index('ParticipantID')

display(tempDF)
display(tempDF.describe(include='all'))

cohortDF = tempDF

# 2. prepare specific test

In [ ]:
#id unique (faster)
tempDF = specificDF.copy()
tempDF = tempDF.set_index('ParticipantID')


tempDF['受診日'] = pd.to_datetime(tempDF['受診日'])

# まず受診日順に並べる
tempDF = tempDF.sort_values('受診日', na_position='last')

# 並べ替え後の状態で重複を削除
tempDF = tempDF.loc[~tempDF.index.duplicated(keep='first')].copy()

print('Before:', len(specificDF))
print('After:', len(tempDF))
print('Unique ID:', tempDF.index.nunique())
print('Duplicated:', tempDF.index.duplicated().sum())

display(tempDF)
display(tempDF.reset_index().describe(include='all'))

specificDF=tempDF

In [ ]:
tempDF1 = specificDF.copy()
tempDF2 = cohortDF

#Clean
tempDF1 = tempDF1.loc[tempDF1.index.isin(tempDF2.index)]

print('specific data:')
display(tempDF1)
display(tempDF1.describe(include='all'))

specificDF = tempDF1#Update

In [ ]:
tempDF2 = specificDF.copy()
tempDF1 = cohortDF

#Clean
tempDF1 = tempDF1.loc[tempDF1.index.isin(tempDF2.index)]

print('cohort data:')
display(tempDF1)
display(tempDF1.describe(include='all'))

cohortDF = tempDF1#Update

In [ ]:
tempDF = cohortDF
variable = 'Age'
xaxis_label = 'Age [years]'
tempD = {'All':'k', 'Female':'tab:red', 'Male':'tab:blue'}

#Check distribution
display(tempDF.groupby('Sex')[variable].describe())
sns.set(style='ticks', context='talk')
plt.figure(figsize=(4, 3))
##Group
for group in tempD.keys():
    if group=='All':
        tempS = tempDF[variable].dropna()
    else:
        tempS = tempDF[variable].loc[tempDF['Sex']==group].dropna()
    print(group+': skewness =', stats.skew(tempS))
    var_label = group+' ('+r'$n$'+' = '+f'{len(tempS):,}'+')'
    sns.distplot(tempS, label=var_label, color=tempD[group])
    plt.axvline(x=tempS.mean(), **{'linestyle':'--', 'color':tempD[group]})
sns.despine()
plt.ylabel('Density')
plt.xlabel(xaxis_label)
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1, labelspacing=0.25)
plt.show()

In [ ]:
tempDF = cohortDF
variable = 'BMI'
xaxis_label = 'BMI [kg m'+r'$^{-2}$'+']'
tempD = {'All':'k', 'Female':'tab:red', 'Male':'tab:blue'}

#Check distribution
display(tempDF.groupby('Sex')[variable].describe())
sns.set(style='ticks',  context='talk')
plt.figure(figsize=(4, 3))
##Group
for group in tempD.keys():
    if group=='All':
        tempS = tempDF[variable].dropna()
    else:
        tempS = tempDF[variable].loc[tempDF['Sex']==group].dropna()
    print(group+': skewness =', stats.skew(tempS))
    var_label = group+' ('+r'$n$'+' = '+f'{len(tempS):,}'+')'
    sns.distplot(tempS, label=var_label, color=tempD[group])
    plt.axvline(x=tempS.mean(), **{'linestyle':'--', 'color':tempD[group]})
sns.despine()
plt.ylabel('Density')
plt.xlabel(xaxis_label)
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1, labelspacing=0.25)
plt.show()

# 3. filter

In [ ]:
tempDF = specificDF.copy()
tempDF = tempDF.drop(columns=spe_dict['administrative'],errors='ignore')
display(tempDF)

specificDF=tempDF

In [ ]:
tempD2={}
tempDF = specificDF.copy()

for category, features in spe_dict.items():

    features = [col for col in features if col in tempDF.columns]

    resultDF = pd.DataFrame({'Feature': features,
                             'N': len(tempDF),
                             'Non-missing N': [tempDF[col].notna().sum() for col in features],
                            'missing N': [tempDF[col].isna().sum() for col in features]})
    resultDF=resultDF.sort_values('Non-missing N',ascending=False)

    print(f'===== {category} =====')
    display(resultDF)

In [ ]:
na_rate = specificDF.isna().mean(axis=1)

print(na_rate.describe())

plt.hist(na_rate, bins=50)
plt.xlabel('Missing rate')
plt.ylabel('N')
plt.show()

specificDFの中で、欠損率が特定の範囲内に入る参加者が何人いるかの図

# 4. drop 0.2(のちにモデルに使う分を残せるように）

In [ ]:
#人数は保持

tempDF = specificDF.copy()

#tempL = (spe_dict['renal'])
#tempD = tempDF.columns.intersection(tempL)

#tempDF = tempDF[tempD].copy()

missing_rate = 0.2
print('DF shape before dropping columns and rows:', tempDF.shape)

#Drop columns with more missingness than cutoff
nrows, ncols = tempDF.shape
tempS = tempDF.isnull().sum(axis=0)
tempS = tempS.loc[tempS <= nrows*missing_rate]
tempDF = tempDF[tempS.index]


print('Baseline DF shape after dropping columns and rows:', tempDF.shape)
display(tempDF)
display(tempDF.describe(include='all'))

speDF = tempDF#Update

In [ ]:
tempDF2 = speDF.copy()
tempDF1 = cohortDF

#Clean
tempDF1 = tempDF1.loc[tempDF1.index.isin(tempDF2.index)]

print('cohort data:')
display(tempDF1)
display(tempDF1.describe(include='all'))

cohortDF = tempDF1#Update

In [ ]:
tempDF = cohortDF
variable = 'Age'
xaxis_label = 'Age [years]'
tempD = {'All':'k', 'Female':'tab:red', 'Male':'tab:blue'}

#Check distribution
display(tempDF.groupby('Sex')[variable].describe())
sns.set(style='ticks', context='talk')
plt.figure(figsize=(4, 3))
##Group
for group in tempD.keys():
    if group=='All':
        tempS = tempDF[variable].dropna()
    else:
        tempS = tempDF[variable].loc[tempDF['Sex']==group].dropna()
    print(group+': skewness =', stats.skew(tempS))
    var_label = group+' ('+r'$n$'+' = '+f'{len(tempS):,}'+')'
    sns.distplot(tempS, label=var_label, color=tempD[group])
    plt.axvline(x=tempS.mean(), **{'linestyle':'--', 'color':tempD[group]})
sns.despine()
plt.ylabel('Density')
plt.xlabel(xaxis_label)
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1, labelspacing=0.25)
plt.show()

In [ ]:
tempDF = cohortDF
variable = 'BMI'
xaxis_label = 'BMI [kg m'+r'$^{-2}$'+']'
tempD = {'All':'k', 'Female':'tab:red', 'Male':'tab:blue'}

#Check distribution
display(tempDF.groupby('Sex')[variable].describe())
sns.set(style='ticks',  context='talk')
plt.figure(figsize=(4, 3))
##Group
for group in tempD.keys():
    if group=='All':
        tempS = tempDF[variable].dropna()
    else:
        tempS = tempDF[variable].loc[tempDF['Sex']==group].dropna()
    print(group+': skewness =', stats.skew(tempS))
    var_label = group+' ('+r'$n$'+' = '+f'{len(tempS):,}'+')'
    sns.distplot(tempS, label=var_label, color=tempD[group])
    plt.axvline(x=tempS.mean(), **{'linestyle':'--', 'color':tempD[group]})
sns.despine()
plt.ylabel('Density')
plt.xlabel(xaxis_label)
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1, labelspacing=0.25)
plt.show()

In [ ]:
tempDF =speDF
for category, features in spe_dict.items():

    features = [col for col in features if col in tempDF.columns]

    resultDF = pd.DataFrame({'Feature': features,
                             'N': len(tempDF),
                             'Non-missing N': [tempDF[col].notna().sum() for col in features],
                            'missing N': [tempDF[col].isna().sum() for col in features]})
    resultDF=resultDF.sort_values('Non-missing N',ascending=False)

    print(f'===== {category} =====')
    display(resultDF)

# 5. transform to less skew

In [ ]:
tempDF = speDF.copy()

#Check skewness
tempL = []
for col_n in tempDF.columns:
    tempS = tempDF[col_n].dropna()
    tempL.append(stats.skew(tempS))
tempS = pd.Series(tempL, index=tempDF.columns, name='Skewness')
tempS = tempS.sort_values(ascending=True)

display(tempS)
display(tempS.describe())

sns.set(style='ticks', context='talk')
plt.figure(figsize=(4, 3))
sns.distplot(tempS)
sns.despine()
plt.ylabel('Density')
plt.xlabel('Skewness')
plt.show()

In [ ]:
#Check negative or zero value prior to transformation
print('Minimum:', tempDF.min(axis=None))

### -1. check types

In [ ]:
tempDF = speDF.copy()

for col in tempDF.columns:
    print(col)
    print('n_unique:', tempDF[col].nunique())
    print('values:', tempDF[col].dropna().unique()[:20])

In [ ]:
tempDF.nunique().sort_values()

赤芽球は全部0なので除外

In [ ]:
#check
sns.histplot(speDF['アルブミン／Cre補正 [mg/g.cre]'], kde=True)

In [ ]:
#check
sns.histplot(speDF['卵白 [UA/mL]'], kde=True)

### 検討

In [ ]:
tempDF = speDF.copy()
tempDF = tempDF.drop(columns='赤芽球 [/100WBC]',errors='ignore')
display(tempDF)

speDF=tempDF

### -3 +1 & log

In [ ]:
tempDF = speDF.copy()
skew_cutoff = 1.5

#Log-transformation if skewness is over cutoff
log_cols = []
for col_n in tempDF.columns:
    tempS = tempDF[col_n].dropna()
    skewness = stats.skew(tempS)
    if skewness > skew_cutoff:
        log_cols.append(col_n)
        tempDF[col_n] = np.log(tempDF[col_n]+1)

display(tempDF)
display(tempDF.describe(include='all'))

speDF = tempDF#Update

In [ ]:
tempDF = speDF

#Check skewness
tempL = []
for col_n in tempDF.columns:
    tempS = tempDF[col_n].dropna()
    tempL.append(stats.skew(tempS))
tempS = pd.Series(tempL, index=tempDF.columns, name='Skewness')
tempS = tempS.sort_values(ascending=True)

display(tempS)
display(tempS.describe())

sns.set(style='ticks',  context='talk')
plt.figure(figsize=(4, 3))
sns.distplot(tempS)
sns.despine()
plt.ylabel('Density')
plt.xlabel('Skewness')
plt.show()

# 6.random forest imputation

In [ ]:
tempDF = speDF

#Check
count = tempDF.isnull().to_numpy().sum(axis=None)
nrows, ncols = tempDF.shape
print('nrows:', nrows)
print('ncols:', ncols)
print('nrows*ncols:', nrows*ncols)
print('nNaNs:', count, '(', count/(nrows*ncols)*100, '[%])')
print('Mean nNaNs per row:', count/nrows)
print('Mean nNaNs per column:', count/ncols)

In [ ]:
tempDF = speDF.copy()

#RF imputation
imputer = MissForest(categorical=None,
                     max_iter=5,
                     rgr=RandomForestRegressor(n_estimators=150,
                                               criterion='squared_error',
                                               max_features='sqrt',
                                               max_depth=None,
                                               n_jobs=-1,
                                               random_state=123))
t_start = time.time()
tempA = imputer.fit_transform(tempDF)
t_elapsed = time.time() - t_start
print('Elapsed time:', round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Clean
tempDF = pd.DataFrame(data=tempA, index=tempDF.index, columns=tempDF.columns)

speDF_RF = tempDF

In [ ]:
tempDF = speDF_RF

#Check
count = tempDF.isnull().to_numpy().sum(axis=None)
nrows, ncols = tempDF.shape
print('nrows:', nrows)
print('ncols:', ncols)
print('nrows*ncols:', nrows*ncols)
print('nNaNs:', count, '(', count/(nrows*ncols)*100, '[%])')
print('Mean nNaNs per row:', count/nrows)
print('Mean nNaNs per column:', count/ncols)

# 7. age related features

In [ ]:
tempDF1 = speDF_RF
tempDF2 = cohortDF


# 対象データ
x = tempDF2['Age']

corr = tempDF1.corrwith(x)

columns = corr.abs().sort_values(ascending=False).index
n_cols = 5
n_rows = math.ceil(len(columns) / n_cols)

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(20, 4 * n_rows))

axes = axes.flatten()

for i, col in enumerate(columns):
    ax = axes[i]

    # 欠損値を除外して描画
    plotDF = pd.DataFrame({
        'Age': x,
        'Feature': tempDF1[col]}).dropna()

    ax.scatter(
        plotDF['Age'],
        plotDF['Feature'],
        alpha=0.5,
        s=10)

    ax.set_title(col, fontsize=10)
    ax.set_xlabel('Age')
    ax.set_ylabel(col)
    ax.grid(alpha=0.2)

# 余ったサブプロットを削除
for j in range(len(columns), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
japanize_matplotlib.japanize()
plt.show()

GFRは年齢を掛け合わせるため不可  
シスタチンCのみは可  

In [ ]:
sns.set(style='ticks', context='notebook')
japanize_matplotlib.japanize()

tempDF = speDF_RF
tempL = (spe_dict['allergy']+spe_dict['kidney'])

# 欠損値を除外
plotDF = tempDF.loc[:,tempDF.columns.isin(tempL)].dropna(axis=0)

# PairGrid
g = sns.PairGrid(plotDF, height=2)

# 左下：散布図
g.map_lower(
    sns.scatterplot,
    alpha=0.5,
    s=15)

# 対角線：分布
g.map_diag(
    sns.histplot,
    kde=True)

# 右上：相関係数とp値
def corrfunc(x, y, **kwargs):
    # 有限値のみ使用
    valid = np.isfinite(x) & np.isfinite(y)
    x = np.asarray(x)[valid]
    y = np.asarray(y)[valid]

    ax = plt.gca()

    if len(x) < 3 or np.std(x) == 0 or np.std(y) == 0:
        ax.text(
            0.5, 0.5, 'N/A',
            ha='center', va='center',
            transform=ax.transAxes)
        return

    r, p = pearsonr(x, y)

    p_text = f'p = {p:.2e}' if p < 0.001 else f'p = {p:.3f}'

    ax.annotate(
        f'r = {r:.3f}\n{p_text}\nn = {len(x)}',
        xy=(0.5, 0.5),
        xycoords='axes fraction',
        ha='center',
        va='center',
        fontsize=10)

    ax.set_axis_off()

g.map_upper(corrfunc)

g.fig.tight_layout()
plt.show()

クロールとナトリウムが0.8以上  
数も同じ  
クロール除外

In [ ]:
#updata distionally

#推定される対応
spe_dict = {
    'hematology': [
        #'桿状核球 [%]',
        #'分葉核球 [%]',
        #'赤芽球 [/100WBC]',
        #'A-Lymphocyte [%]',
        #'Pro-Myelo [%]',
        #'Myeloblast [%]',
        #'Myelo [%]',
        #'Meta-Myelo [%]'
    ],

    'clinical_test': [
        #'NT-proBNP',
        #'グリコアルブミン [%]',
        'ペプシノーゲンⅠ [ng/mL]',
        'ペプシノーゲンⅡ [ng/mL]',
        'PG1/PG2 比 ',
        #'高感度心筋トロポニンT [ng/mL]'
        ],

    'allergy': [
        # アレルゲン関連項目
        'オオアワガエリ [LC]',
        'ハルガヤ [LC]',
        'カモガヤ [LC]',
        'カモガヤ [UA/mL]', 'ブタクサ混合物Ⅰ [LC]', 'ヨモギ [LC]', 'ヨモギ [UA/mL] ',
        'スギ [LC]', 'スギ [UA/mL] ', 'スギ [UA/mL] 岩手', 'ヒノキ [LC]',
        'ハンノキ [LC]', 'シラカンバ [LC]', 'コナヒョウヒダニ [LC]', 
        'コナヒョウヒダニ [UA/mL] ', 'コナヒョウヒダニ [UA/mL] 岩手', 'ハウスダストⅠ [LC]', 
        'ペニシリウム [LC]', 'クラドスポリウム [LC]', 'カンジダ [LC]', 'カンジダ [UA/mL]', 
        'アルテルナリア [LC]', 'アスペルギルス [LC]', 'ネコ皮屑 [LC]', 'ネコ皮屑 [UA/mL]', 
        'イヌ皮屑 [LC]', '小麦 [LC]', '大豆 [LC]', '米 [LC]', 'マグロ [LC]', 'サケ [LC]', 
        'エビ [LC]', 'エビ [UA/mL]', 'カニ [LC]', 'チェダーチーズ [LC]', 'ミルク [LC]', 
        '牛肉 [LC]', '鶏肉 [LC]', '卵白 [LC]', '卵白 [UA/mL]', 'ソバ [LC]', 
        'ピーナッツ [LC]', 'ラテックス [LC]',
        '抗体濃度 [U/mL]',
        'IgE（非特異的IgE） [IU/mL]'],

    'renal': [
        'アルブミン／Cre補正 [mg/g.cre]',
        #'アルブミン定量値 [μg/mL]',
        'クレアチニン（尿中） [g/L]',
        'Na（ナトリウム）（尿中） [g/L]',
        #'Cl（クロール）（尿中） [g/L]',
        'K（カリウム）（尿中） [g/L]',
        'シスタチンC［金コロイド凝集法］ [mg/L]',
        #'GFR区分（シスタチンC）[ml/min/1.73㎡]'
    ],
    
    'administrative': [
        '受診日',
        'イベントコード',
        'コースコード','検査会社']}

# 8. save

## -1 allergy model

In [ ]:
#Save data with NAs
fileDir = './ExportData/Preprocess/labo/'
ipynbName = '261005_TMM-BAge_'
fileName = 'AlgAge_laboDF-MF.tsv'


tempL = (spe_dict['allergy'])
tempD = speDF_RF.columns.intersection(tempL)

tempDF = speDF_RF[tempD].copy()
tempDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)

display(tempDF)
display(tempDF.describe(include='all'))

## -2 renal

In [ ]:
#Save data with NAs
fileDir = './ExportData/Preprocess/labo/'
ipynbName = '261005_TMM-BAge_'
fileName = 'KidAge_laboDF-MF.tsv'


tempL = (spe_dict['renal'])
tempD = speDF_RF.columns.intersection(tempL)

tempDF = speDF_RF[tempD].copy()
tempDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)

display(tempDF)
display(tempDF.describe(include='all'))

## -3 all

In [ ]:
#Save data with NAs
fileDir = './ExportData/Preprocess/labo/'
ipynbName = '261005_TMM-BAge_'
fileName = 'laboDF-MF.tsv'

tempDF = specificDF.copy()
tempDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)

display(tempDF)
display(tempDF.describe(include='all'))

# 9. set validation

## import

In [ ]:
#validation participants
fileDir = './ExportData/Preprocess/metabolome/'
ipynbName = '260924_TMM-BAge_PreprocessMetDF-CommCohort67K-ver1_'
fileName = 'demographicsDF_1k.tsv'
tempDF = pd.read_csv(fileDir+ipynbName+fileName, sep='\t')
tempDF = tempDF.set_index('ParticipantID')

display(tempDF)
display(tempDF.describe(include='all'))

cohortDF = tempDF 

In [ ]:
# laboratory data

fileDir = './ExportData/Preprocess/'
ipynbName = '260916_TMM-BAge_CommCohort67K-ver1_'
fileName = 'laboDF-rename-date.tsv'
tempDF = pd.read_csv(fileDir+ipynbName+fileName, sep='\t')

display(tempDF)
display(tempDF.describe(include='all'))
labDF=tempDF #laboratory

In [ ]:
#id unique (faster)
tempDF = labDF.copy()
tempDF = tempDF.set_index('ParticipantID')


tempDF['受診日'] = pd.to_datetime(tempDF['受診日'])

# まず受診日順に並べる
tempDF = tempDF.sort_values('受診日', na_position='last')

# 並べ替え後の状態で重複を削除
tempDF = tempDF.loc[~tempDF.index.duplicated(keep='first')].copy()

print('Before:', len(specificDF))
print('After:', len(tempDF))
print('Unique ID:', tempDF.index.nunique())
print('Duplicated:', tempDF.index.duplicated().sum())

display(tempDF)
display(tempDF.reset_index().describe(include='all'))

labDF=tempDF

## 2. prepare dataframe

### -1 slice paraticipants

In [ ]:
tempDF1 = labDF.copy()
tempDF2 = cohortDF

#Clean
tempDF1 = tempDF1.loc[tempDF1.index.isin(tempDF2.index)]

print('specific data:')
display(tempDF1)
display(tempDF1.describe(include='all'))

labDF = tempDF1#Update

### -2 define dict

In [ ]:
#推定される対応
lab_dict = {
    'hematology': [
        '桿状核球 [%]',
        '分葉核球 [%]',
        '赤芽球 [/100WBC]',
        'A-Lymphocyte [%]',
        'Pro-Myelo [%]',
        'Myeloblast [%]',
        'Myelo [%]',
        'Meta-Myelo [%]'],

    'clinical_test': [
        'NT-proBNP',
        'グリコアルブミン [%]',
        'ペプシノーゲンⅠ [ng/mL]',
        'ペプシノーゲンⅡ [ng/mL]',
        'PG1/PG2 比 ',
        '高感度心筋トロポニンT [ng/mL]'],

    'allergy': [
        # アレルゲン関連項目
        'オオアワガエリ [LC]',
        'ハルガヤ [LC]',
        'カモガヤ [LC]',
        'カモガヤ [UA/mL]', 'ブタクサ混合物Ⅰ [LC]', 'ヨモギ [LC]', 'ヨモギ [UA/mL] ',
        'スギ [LC]', 'スギ [UA/mL] ', 'スギ [UA/mL] 岩手', 'ヒノキ [LC]',
        'ハンノキ [LC]', 'シラカンバ [LC]', 'コナヒョウヒダニ [LC]', 
        'コナヒョウヒダニ [UA/mL] ', 'コナヒョウヒダニ [UA/mL] 岩手', 'ハウスダストⅠ [LC]', 
        'ペニシリウム [LC]', 'クラドスポリウム [LC]', 'カンジダ [LC]', 'カンジダ [UA/mL]', 
        'アルテルナリア [LC]', 'アスペルギルス [LC]', 'ネコ皮屑 [LC]', 'ネコ皮屑 [UA/mL]', 
        'イヌ皮屑 [LC]', '小麦 [LC]', '大豆 [LC]', '米 [LC]', 'マグロ [LC]', 'サケ [LC]', 
        'エビ [LC]', 'エビ [UA/mL]', 'カニ [LC]', 'チェダーチーズ [LC]', 'ミルク [LC]', 
        '牛肉 [LC]', '鶏肉 [LC]', '卵白 [LC]', '卵白 [UA/mL]', 'ソバ [LC]', 
        'ピーナッツ [LC]', 'ラテックス [LC]',
        '抗体濃度 [U/mL]',
        'IgE（非特異的IgE） [IU/mL]'],

    'kidney': [
        'アルブミン／Cre補正 [mg/g.cre]',
        'アルブミン定量値 [μg/mL]',
        'クレアチニン（尿中） [g/L]',
        'Na（ナトリウム）（尿中） [g/L]',
        'Cl（クロール）（尿中） [g/L]',
        'K（カリウム）（尿中） [g/L]',
        'シスタチンC［金コロイド凝集法］ [mg/L]',
        'GFR区分（シスタチンC）[ml/min/1.73㎡]'],
    
    'administrative': [
        '受診日',
        'イベントコード',
        'コースコード','検査会社']}

## 3. check 

In [ ]:
tempD2={}
tempDF = labDF.copy()

for category, features in lab_dict.items():

    features = [col for col in features if col in tempDF.columns]

    resultDF = pd.DataFrame({'Feature': features,
                             'N': len(tempDF),
                             'Non-missing N': [tempDF[col].notna().sum() for col in features],
                            'missing N': [tempDF[col].isna().sum() for col in features]})
    resultDF=resultDF.sort_values('Non-missing N',ascending=False)

    print(f'===== {category} =====')
    display(resultDF)

## 4. make validation

### -1 slice

In [ ]:
tempDF1 = labDF.copy()
tempDF2 = speDF

#Clean
tempDF1 = tempDF1.loc[:,tempDF1.columns.isin(tempDF2.columns)]

print('cohort data:')
display(tempDF1)
display(tempDF1.describe(include='all'))

laboDF = tempDF1#Update

### -2 skewness

In [ ]:
tempDF = laboDF

#Check skewness
tempL = []
for col_n in tempDF.columns:
    tempS = tempDF[col_n].dropna()
    tempL.append(stats.skew(tempS))
tempS = pd.Series(tempL, index=tempDF.columns, name='Skewness')
tempS = tempS.sort_values(ascending=True)

display(tempS)
display(tempS.describe())

sns.set(style='ticks', context='talk')
plt.figure(figsize=(4, 3))
sns.distplot(tempS)
sns.despine()
plt.ylabel('Density')
plt.xlabel('Skewness')
plt.show()

In [ ]:
#Check negative or zero value prior to transformation
print('Minimum:', laboDF.min(axis=None))

In [ ]:
tempDF = laboDF.copy()

# use dicsovery
tempDF[log_cols] = np.log(tempDF[log_cols]+1)

display(tempDF)
display(tempDF.describe(include='all'))

laboDF = tempDF#Update

In [ ]:
tempDF = laboDF

#Check skewness
tempL = []
for col_n in tempDF.columns:
    tempS = tempDF[col_n].dropna()
    tempL.append(stats.skew(tempS))
tempS = pd.Series(tempL, index=tempDF.columns, name='Skewness')
tempS = tempS.sort_values(ascending=True)

display(tempS)
display(tempS.describe())

sns.set(style='ticks',  context='talk')
plt.figure(figsize=(4, 3))
sns.distplot(tempS)
sns.despine()
plt.ylabel('Density')
plt.xlabel('Skewness')
plt.show()

### -2 MF

In [ ]:
na_rate = laboDF.isna().mean(axis=1)

print(na_rate.describe())

plt.hist(na_rate, bins=50)
plt.xlabel('Missing rate')
plt.ylabel('N')
plt.show()

In [ ]:
tempDF = laboDF

#Check
count = tempDF.isnull().to_numpy().sum(axis=None)
nrows, ncols = tempDF.shape
print('nrows:', nrows)
print('ncols:', ncols)
print('nrows*ncols:', nrows*ncols)
print('nNaNs:', count, '(', count/(nrows*ncols)*100, '[%])')
print('Mean nNaNs per row:', count/nrows)
print('Mean nNaNs per column:', count/ncols)

In [ ]:
tempDF = laboDF.copy()

t_start = time.time()
tempA = imputer.transform(tempDF)
t_elapsed = time.time() - t_start
print('Elapsed time:', round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Clean
tempDF = pd.DataFrame(data=tempA, index=tempDF.index, columns=tempDF.columns)

laboDF_RF = tempDF

In [ ]:
tempDF = laboDF_RF

#Check
count = tempDF.isnull().to_numpy().sum(axis=None)
nrows, ncols = tempDF.shape
print('nrows:', nrows)
print('ncols:', ncols)
print('nrows*ncols:', nrows*ncols)
print('nNaNs:', count, '(', count/(nrows*ncols)*100, '[%])')
print('Mean nNaNs per row:', count/nrows)
print('Mean nNaNs per column:', count/ncols)

## 5. save

In [ ]:
#推定される対応
lab_dict = {
    'hematology': [
        '桿状核球 [%]',
        '分葉核球 [%]',
        '赤芽球 [/100WBC]',
        'A-Lymphocyte [%]',
        'Pro-Myelo [%]',
        'Myeloblast [%]',
        'Myelo [%]',
        'Meta-Myelo [%]'],

    'clinical_test': [
        'NT-proBNP',
        'グリコアルブミン [%]',
        'ペプシノーゲンⅠ [ng/mL]',
        'ペプシノーゲンⅡ [ng/mL]',
        'PG1/PG2 比 ',
        '高感度心筋トロポニンT [ng/mL]'],

    'allergy': [
        # アレルゲン関連項目
        'オオアワガエリ [LC]',
        'ハルガヤ [LC]',
        'カモガヤ [LC]',
        'カモガヤ [UA/mL]', 'ブタクサ混合物Ⅰ [LC]', 'ヨモギ [LC]', 'ヨモギ [UA/mL] ',
        'スギ [LC]', 'スギ [UA/mL] ', 'スギ [UA/mL] 岩手', 'ヒノキ [LC]',
        'ハンノキ [LC]', 'シラカンバ [LC]', 'コナヒョウヒダニ [LC]', 
        'コナヒョウヒダニ [UA/mL] ', 'コナヒョウヒダニ [UA/mL] 岩手', 'ハウスダストⅠ [LC]', 
        'ペニシリウム [LC]', 'クラドスポリウム [LC]', 'カンジダ [LC]', 'カンジダ [UA/mL]', 
        'アルテルナリア [LC]', 'アスペルギルス [LC]', 'ネコ皮屑 [LC]', 'ネコ皮屑 [UA/mL]', 
        'イヌ皮屑 [LC]', '小麦 [LC]', '大豆 [LC]', '米 [LC]', 'マグロ [LC]', 'サケ [LC]', 
        'エビ [LC]', 'エビ [UA/mL]', 'カニ [LC]', 'チェダーチーズ [LC]', 'ミルク [LC]', 
        '牛肉 [LC]', '鶏肉 [LC]', '卵白 [LC]', '卵白 [UA/mL]', 'ソバ [LC]', 
        'ピーナッツ [LC]', 'ラテックス [LC]',
        '抗体濃度 [U/mL]',
        'IgE（非特異的IgE） [IU/mL]'],

    'kidney': [
        'アルブミン／Cre補正 [mg/g.cre]',
        #'アルブミン定量値 [μg/mL]',
        'クレアチニン（尿中） [g/L]',
        'Na（ナトリウム）（尿中） [g/L]',
        #'Cl（クロール）（尿中） [g/L]',
        'K（カリウム）（尿中） [g/L]',
        'シスタチンC［金コロイド凝集法］ [mg/L]',
        #'GFR区分（シスタチンC）[ml/min/1.73㎡]'
    ],
    
    'administrative': [
        '受診日',
        'イベントコード',
        'コースコード','検査会社']}

### -1 alg

In [ ]:
#Save data with NAs
fileDir = './ExportData/Preprocess/labo/'
ipynbName = '261005_TMM-BAge_'
fileName = 'AlgAge_validation-MF.tsv'


tempL = (lab_dict['allergy'])
tempD = laboDF_RF.columns.intersection(tempL)

tempDF = laboDF_RF[tempD].copy()
tempDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)

display(tempDF)
display(tempDF.describe(include='all'))

### -2 kid

In [ ]:
#Save data with NAs
fileDir = './ExportData/Preprocess/labo/'
ipynbName = '261005_TMM-BAge_'
fileName = 'KidAge_validation-MF.tsv'


tempL = (lab_dict['kidney'])
tempD = laboDF_RF.columns.intersection(tempL)

tempDF = laboDF_RF[tempD].copy()
tempDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)

display(tempDF)
display(tempDF.describe(include='all'))

### -3 all

In [ ]:
#Save data with NAs
fileDir = './ExportData/Preprocess/labo/'
ipynbName = '261005_TMM-BAge_'
fileName = 'validation-MF.tsv'

tempDF = labDF.copy()
tempDF = tempDF.drop(columns=lab_dict['administrative'],errors='ignore')

tempDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)

display(tempDF)
display(tempDF.describe(include='all'))

# end of notebook